# WeightedKgBlend — CBR
Simple Case-Based Reasoning (Das et al. 2020). Reads splits from `weightedkgblend-splits`.


In [ ]:
from pathlib import Path
import pandas as pd, time
from collections import defaultdict

INDICATION_REL = 'indication'
SPLITS_TO_RUN  = [0]   # change to list(range(5)) for all splits

SPLITS = Path('/kaggle/input/weightedkgblend-splits/splits')
WORK   = Path('/kaggle/working')
WORK.mkdir(exist_ok=True)

if not SPLITS.exists():
    raise FileNotFoundError(f"Splits not found at {SPLITS}")

print(f'Splits to run : {SPLITS_TO_RUN}')
print(f'Splits found  : {sorted([s.name for s in SPLITS.iterdir()])}')


In [ ]:
def build_graph(tsv):
    df = pd.read_csv(tsv, sep='\t', header=None, names=['h','r','t'])
    g  = defaultdict(set)
    for _, row in df.iterrows():
        g[row.h].add((row.r, row.t))
        g[row.t].add((f'inv_{row.r}', row.h))
    return dict(g)

def cbr_predict(drug, graph, train_triples, k=100):
    """
    Returns:
      scores       : dict {disease: weighted_vote}
      top_neighbor : dict {disease: best_contributing_neighbor}
    """
    q_edges   = graph.get(drug, set())
    nb_scores = {}
    for h, r, t in train_triples:
        if h == drug: continue
        overlap = len(q_edges & graph.get(h, set()))
        if overlap: nb_scores[h] = nb_scores.get(h, 0) + overlap
    top_nb = sorted(nb_scores, key=nb_scores.get, reverse=True)[:k]
    total  = sum(nb_scores.get(n, 1) for n in top_nb) or 1
    scores       = defaultdict(float)
    top_neighbor = {}   # disease -> best contributing neighbor
    for nb in top_nb:
        w = nb_scores.get(nb, 1) / total
        for h, r, t in train_triples:
            if h == nb and r == INDICATION_REL:
                scores[t] += w
                if t not in top_neighbor or nb_scores[nb] > nb_scores.get(top_neighbor[t], 0):
                    top_neighbor[t] = nb
    return dict(scores), top_neighbor

def format_path(drug, neighbor, disease):
    return f"{drug} --[similar_to]--> {neighbor} --[indication]--> {disease}"

In [ ]:
PRED_DIR = WORK / 'predictions' / 'CBR'
total_start = time.time()

for i in SPLITS_TO_RUN:
    sl     = SPLITS / f'slice_{i}'
    outdir = PRED_DIR / f'slice_{i}'
    outdir.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    print(f'\n{"="*50}\nCBR — slice_{i}\n{"="*50}')

    tr_triples = [(r.h,r.r,r.t) for r in
        pd.read_csv(sl/'ind_train.tsv', sep='\t', header=None, names=['h','r','t']).itertuples()]
    graph  = build_graph(sl/'kge_train.tsv')
    n_ents = len(pd.read_csv(sl/'entities.txt', header=None))

    for split in ['test', 'valid']:
        done = outdir / f'predictions_{split}.tsv'
        if done.exists(): print(f'  SKIP {split}'); continue
        ev = [(r.h,r.r,r.t) for r in
              pd.read_csv(sl/f'ind_{split}.tsv', sep='\t', header=None, names=['h','r','t']).itertuples()]
        rows = []
        for drug, rel, exp_dis in ev:
            scores, top_neighbor = cbr_predict(drug, graph, tr_triples)
            if not scores:
                rank = n_ents
                path = ''
            else:
                sd   = sorted(scores, key=scores.get, reverse=True)
                rank = sd.index(exp_dis) + 1 if exp_dis in sd else n_ents
                # Path for the top-ranked prediction
                top_disease = sd[0]
                nb = top_neighbor.get(top_disease, '')
                path = format_path(drug, nb, top_disease) if nb else ''
            rows.append({'drug': drug, 'expected_disease': exp_dis,
                         'rank': rank, 'reciprocal_rank': 1.0/rank,
                         'top_path': path})
        df_out = pd.DataFrame(rows)
        df_out.to_csv(done, sep='\t', index=False)
        print(f'  {split} MRR = {df_out.reciprocal_rank.mean():.4f}')
        print(f'  Example path: {df_out.top_path.iloc[0]}')

    elapsed = time.time() - t0
    print(f'\n  slice_{i} done in {elapsed/60:.1f} min')

total = time.time() - total_start
print(f'\nTotal: {total/60:.1f} min')
print(f'Estimated for all 5 splits: {total/60 * 5 / len(SPLITS_TO_RUN):.1f} min')